# Step 01_04_06 -- old_rating PRE-GAME Temporal Audit

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_04 -- Data Cleaning
**Step:** 01_04_06
**Dataset:** aoestats
**Predecessors:** 01_04_01 (duplicate policy, profile_id cast), 01_04_02 (`matches_1v1_clean`),
  01_04_03 (`matches_history_minimal`), 01_03_01 (PRE-GAME deferral flag)
**Invariants touched:** I3, I6, I7

**Purpose:** Close the `old_rating` PRE-GAME deferral flag in `INVARIANTS.md §3`
(currently "Formal bivariate temporal leakage test deferred to Phase 02").
The empirical test: for consecutive matches t, t+1 of the same `profile_id` on the
**same leaderboard**, `players_raw.old_rating[t+1]` should equal
`players_raw.new_rating[t]`. High agreement confirms the API honours the
old/new-rating convention and that `old_rating` is the pre-match rating state.

**Critical methodological note (BLOCKER 1 fix):**
AoE2 ratings are per-leaderboard independent systems. The LAG window MUST be
`PARTITION BY (profile_id_i64, leaderboard)`. A naive global partition across
leaderboards yields cross-leaderboard pairs whose disagreement is a feature of
independent rating systems, not an API convention violation.

**CAST discipline (BLOCKER 2 / DS-AOESTATS-IDENTITY-04):**
`profile_id` is stored as DOUBLE in `players_raw` (mixed-type ingestion).
All values are below 2^53 so `CAST(profile_id AS BIGINT)` is lossless.
Defined as `profile_id_i64` in every CTE before partition-key use.

**I6 discipline:** All SQL that produces reported results appears verbatim below.

**Exploration only (I9):** No VIEWs created, no raw tables modified.
**Date:** 2026-04-21

## Hypothesis and Falsifier

**H0 (primary scope `leaderboard = 'random_map'`):**
For consecutive matches t, t+1 of the same `profile_id` ordered by
`(started_timestamp, game_id)` within the same leaderboard,
`players_raw.old_rating[t+1] == players_raw.new_rating[t]` at
`agreement_rate >= 0.95` AND `max(|old_rating[t+1] - new_rating[t]|) < 50`
rating units. Per-leaderboard strata pass at >= 0.90.
Per-time-gap-bucket strata pass at >= 0.90.

**Falsifier:** Any of the three gates fails → FAIL.
Catastrophic: `agreement_rate < 0.80` → HALT, escalate to user.

**Threshold argument (I7):**
- 5% disagreement ceiling: DS-AOESTATS-02 documents NULLIF cleaning-loss of
  ~0.03% for `old_rating=0` sentinel rows. A disagreement rate > 5% would mean
  >= 5% of `old_rating` values are potentially POST-GAME contaminated -- exceeding
  the established cleaning-loss tolerance by 2 orders of magnitude. The 5% ceiling
  is the threshold at which the PRE-GAME classification becomes materially unsound.
- 50-rating-unit disagreement magnitude: Elo K-factor for aoestats is approximately
  20-40 rating points per match. Disagreements larger than 50 units indicate
  systematic drift (rating resets, season resets, or API semantic drift) beyond
  normal per-match Elo updates. The 50-unit threshold is ~1.25x the K=40 upper bound.

## Cell 1 -- Imports

In [1]:
import json
from datetime import date
from pathlib import Path

from scipy.stats import binomtest

from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()
print("Imports complete.")

Imports complete.


## Cell 2 -- DuckDB Connection (read-only)

Read-only per sandbox contract and I9 (exploration only).

In [2]:
db = get_notebook_db("aoe2", "aoestats", read_only=True)
con = db.con
duckdb_version = con.execute("SELECT version()").fetchone()[0]
print(f"DuckDB connected (read-only). Version: {duckdb_version}")

DuckDB connected (read-only). Version: v1.5.1


## Cell 3 -- Reports Directory + Artifact Paths

In [3]:
reports_dir = get_reports_dir("aoe2", "aoestats")
artifacts_dir = reports_dir / "artifacts" / "01_exploration" / "04_cleaning"
artifacts_dir.mkdir(parents=True, exist_ok=True)
json_path = artifacts_dir / "01_04_06_old_rating_temporal_audit.json"
md_path = artifacts_dir / "01_04_06_old_rating_temporal_audit.md"
print(f"Artifacts target: {artifacts_dir}")

Artifacts target: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/04_cleaning


## Cell 4 -- Schema Validation

Assert all required columns exist before proceeding.

In [4]:
SQL_SCHEMA_PLAYERS = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'players_raw'
  AND column_name IN ('old_rating', 'new_rating', 'profile_id', 'game_id')
ORDER BY column_name
"""

SQL_SCHEMA_MATCHES = """
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'matches_raw'
  AND column_name IN ('leaderboard', 'game_id', 'started_timestamp')
ORDER BY column_name
"""

players_cols = {r[0] for r in con.execute(SQL_SCHEMA_PLAYERS).fetchall()}
matches_cols = {r[0] for r in con.execute(SQL_SCHEMA_MATCHES).fetchall()}

required_players = {"old_rating", "new_rating", "profile_id", "game_id"}
required_matches = {"leaderboard", "game_id", "started_timestamp"}

assert required_players <= players_cols, f"Missing players_raw columns: {required_players - players_cols}"
assert required_matches <= matches_cols, f"Missing matches_raw columns: {required_matches - matches_cols}"
print("Schema validation PASSED.")
print(f"  players_raw columns confirmed: {sorted(required_players)}")
print(f"  matches_raw columns confirmed: {sorted(required_matches)}")

Schema validation PASSED.
  players_raw columns confirmed: ['game_id', 'new_rating', 'old_rating', 'profile_id']
  matches_raw columns confirmed: ['game_id', 'leaderboard', 'started_timestamp']


## Analysis §1 -- Tie-Rate + Duplicate-Row Pre-Flight

Measure tie-rate on `(profile_id_i64, leaderboard, started_timestamp)` and
count `(game_id, profile_id)` duplicates (expected 489 per INVARIANTS §1).
If tie rate > 1%, halt and revise filter strategy.

In [5]:
SQL_TIE_RATE = """
WITH base AS (
  SELECT
    CAST(p.profile_id AS BIGINT) AS profile_id_i64,
    m.leaderboard,
    m.started_timestamp
  FROM players_raw p
  JOIN matches_raw m USING (game_id)
  WHERE p.profile_id IS NOT NULL
    AND p.old_rating IS NOT NULL
    AND p.new_rating IS NOT NULL
),
tie_check AS (
  SELECT profile_id_i64, leaderboard, started_timestamp, COUNT(*) AS cnt
  FROM base
  GROUP BY profile_id_i64, leaderboard, started_timestamp
)
SELECT
  COUNT(*)                                       AS total_groups,
  COUNT(*) FILTER (WHERE cnt > 1)                AS tied_groups,
  ROUND(
    COUNT(*) FILTER (WHERE cnt > 1) * 100.0 / COUNT(*),
    6
  )                                              AS tie_rate_pct
FROM tie_check
"""

tie_result = con.execute(SQL_TIE_RATE).fetchone()
total_groups, tied_groups, tie_rate_pct = tie_result
tie_rate = float(tied_groups) / float(total_groups) if total_groups > 0 else 0.0

print(f"Tie-rate check (profile_id_i64, leaderboard, started_timestamp):")
print(f"  Total groups: {total_groups:,}")
print(f"  Tied groups:  {tied_groups:,}")
print(f"  Tie rate:     {tie_rate_pct}%")

if tie_rate > 0.01:
    raise RuntimeError(
        f"Tie rate {tie_rate:.4%} > 1% threshold. "
        "Revise filter strategy before proceeding."
    )
print("  Tie rate <= 1%: OK to proceed with (started_timestamp, game_id) ordering.")

Tie-rate check (profile_id_i64, leaderboard, started_timestamp):
  Total groups: 107,626,399
  Tied groups:  0
  Tie rate:     0.0%
  Tie rate <= 1%: OK to proceed with (started_timestamp, game_id) ordering.


In [6]:
SQL_DUPLICATES = """
SELECT
  COUNT(*) FILTER (WHERE cnt > 1)      AS dup_groups,
  COALESCE(
    SUM(cnt) FILTER (WHERE cnt > 1) - COUNT(*) FILTER (WHERE cnt > 1),
    0
  )                                    AS extra_rows
FROM (
  SELECT game_id, CAST(profile_id AS BIGINT) AS profile_id_i64, COUNT(*) AS cnt
  FROM players_raw
  WHERE profile_id IS NOT NULL
  GROUP BY game_id, CAST(profile_id AS BIGINT)
)
"""

dup_result = con.execute(SQL_DUPLICATES).fetchone()
dup_groups, extra_rows = dup_result
print(f"Duplicate (game_id, profile_id_i64) check:")
print(f"  Duplicate groups: {dup_groups}")
print(f"  Extra rows (to collapse via DISTINCT): {extra_rows}")
# Note: expected 489 duplicates per INVARIANTS §1 refers to duplicate *rows* in
# players_raw before deduplication. If extra_rows = 0 here, it means the
# profile_id DOUBLE → BIGINT cast resolves the apparent duplicates (they were
# floating-point variants of the same integer). Either way, DISTINCT handles it.
print("  DISTINCT on (game_id, profile_id_i64) applied in main CTE for safety.")

Duplicate (game_id, profile_id_i64) check:
  Duplicate groups: 0
  Extra rows (to collapse via DISTINCT): 0
  DISTINCT on (game_id, profile_id_i64) applied in main CTE for safety.


## Analysis §2 -- Pair Construction (Leaderboard-Partitioned)

Core SQL using `players_raw JOIN matches_raw USING (game_id)`.
LAG window: `PARTITION BY (profile_id_i64, leaderboard) ORDER BY (started_timestamp, game_id)`.
DISTINCT on `(game_id, profile_id_i64)` before windowing to collapse any duplicates.
Pair count reported overall and per leaderboard.

In [7]:
SQL_PAIRS_PER_LEADERBOARD = """
WITH deduped AS (
  SELECT DISTINCT
    CAST(p.profile_id AS BIGINT) AS profile_id_i64,
    m.leaderboard,
    m.started_timestamp,
    m.game_id,
    p.old_rating,
    p.new_rating
  FROM players_raw p
  JOIN matches_raw m USING (game_id)
  WHERE p.profile_id IS NOT NULL
    AND p.old_rating IS NOT NULL
    AND p.new_rating IS NOT NULL
),
with_lag AS (
  SELECT
    profile_id_i64,
    leaderboard,
    started_timestamp,
    game_id,
    old_rating,
    new_rating,
    LAG(new_rating) OVER (
      PARTITION BY profile_id_i64, leaderboard
      ORDER BY started_timestamp, game_id
    ) AS prev_new_rating,
    LAG(started_timestamp) OVER (
      PARTITION BY profile_id_i64, leaderboard
      ORDER BY started_timestamp, game_id
    ) AS prev_started_timestamp
  FROM deduped
),
pairs AS (
  SELECT *
  FROM with_lag
  WHERE prev_new_rating IS NOT NULL
)
SELECT
  leaderboard,
  COUNT(*)                                              AS n_pairs,
  AVG(CASE WHEN old_rating = prev_new_rating THEN 1.0 ELSE 0.0 END)
                                                        AS agreement_rate,
  MAX(ABS(old_rating - prev_new_rating))                AS max_disagreement
FROM pairs
GROUP BY leaderboard
ORDER BY n_pairs DESC
"""

lb_result = con.execute(SQL_PAIRS_PER_LEADERBOARD).df()
print("Per-leaderboard pair counts and agreement rates:")
print(lb_result.to_string(index=False))

n_pairs_total = int(lb_result["n_pairs"].sum())
print(f"\nTotal pairs across all leaderboards: {n_pairs_total:,}")

Per-leaderboard pair counts and agreement rates:
       leaderboard  n_pairs  agreement_rate  max_disagreement
   team_random_map 67428909        0.860128               772
        random_map 35275197        0.920968              1118
co_team_random_map  2783734        0.786743               490
     co_random_map  1212477        0.855239               720

Total pairs across all leaderboards: 106,700,317


## Analysis §3 -- Agreement + Disagreement-Magnitude (Primary Scope: random_map)

In [8]:
SQL_PRIMARY_AGREEMENT = """
WITH deduped AS (
  SELECT DISTINCT
    CAST(p.profile_id AS BIGINT) AS profile_id_i64,
    m.leaderboard,
    m.started_timestamp,
    m.game_id,
    p.old_rating,
    p.new_rating
  FROM players_raw p
  JOIN matches_raw m USING (game_id)
  WHERE p.profile_id IS NOT NULL
    AND p.old_rating IS NOT NULL
    AND p.new_rating IS NOT NULL
),
with_lag AS (
  SELECT
    profile_id_i64,
    leaderboard,
    started_timestamp,
    game_id,
    old_rating,
    new_rating,
    LAG(new_rating) OVER (
      PARTITION BY profile_id_i64, leaderboard
      ORDER BY started_timestamp, game_id
    ) AS prev_new_rating,
    LAG(started_timestamp) OVER (
      PARTITION BY profile_id_i64, leaderboard
      ORDER BY started_timestamp, game_id
    ) AS prev_started_timestamp
  FROM deduped
),
pairs AS (
  SELECT *,
    CASE WHEN old_rating = prev_new_rating THEN 1 ELSE 0 END AS agreed,
    ABS(old_rating - prev_new_rating)                         AS disagreement_abs
  FROM with_lag
  WHERE prev_new_rating IS NOT NULL
    AND leaderboard = 'random_map'
)
SELECT
  COUNT(*)                                                          AS n_pairs,
  SUM(agreed)                                                       AS n_agreed,
  AVG(agreed::DOUBLE)                                               AS agreement_rate,
  MAX(disagreement_abs)                                             AS max_disagreement,
  MEDIAN(CASE WHEN agreed = 0 THEN disagreement_abs ELSE NULL END)  AS median_dis_disagreeing
FROM pairs
"""

primary = con.execute(SQL_PRIMARY_AGREEMENT).fetchone()
n_pairs_primary = int(primary[0])
n_agreed_primary = int(primary[1])
agreement_rate_primary = float(primary[2])
max_disagreement_primary = int(primary[3])
median_dis_disagreeing = float(primary[4]) if primary[4] is not None else None

print(f"Primary scope (random_map):")
print(f"  n_pairs:           {n_pairs_primary:,}")
print(f"  n_agreed:          {n_agreed_primary:,}")
print(f"  agreement_rate:    {agreement_rate_primary:.6f}")
print(f"  max_disagreement:  {max_disagreement_primary} rating units")
print(f"  median_dis (disagreeing only): {median_dis_disagreeing}")

# Wilson CI
binom_result = binomtest(n_agreed_primary, n_pairs_primary)
ci = binom_result.proportion_ci(confidence_level=0.95, method="wilson")
wilson_ci_low = float(ci.low)
wilson_ci_high = float(ci.high)
print(f"  Wilson CI 95%:     [{wilson_ci_low:.6f}, {wilson_ci_high:.6f}]")

Primary scope (random_map):
  n_pairs:           35,275,197
  n_agreed:          32,487,342
  agreement_rate:    0.920968
  max_disagreement:  1118 rating units
  median_dis (disagreeing only): 16.0
  Wilson CI 95%:     [0.920879, 0.921057]


## Analysis §4 -- Stratification (Per Leaderboard + Per Time-Gap Bucket)

Per-leaderboard: all distinct leaderboards. Per-time-gap-bucket on random_map.
PASS requires ALL strata >= 0.90 (relaxed threshold for sub-strata).

In [9]:
SQL_TIME_GAP_STRATIFICATION = """
WITH deduped AS (
  SELECT DISTINCT
    CAST(p.profile_id AS BIGINT) AS profile_id_i64,
    m.leaderboard,
    m.started_timestamp,
    m.game_id,
    p.old_rating,
    p.new_rating
  FROM players_raw p
  JOIN matches_raw m USING (game_id)
  WHERE p.profile_id IS NOT NULL
    AND p.old_rating IS NOT NULL
    AND p.new_rating IS NOT NULL
    AND m.leaderboard = 'random_map'
),
with_lag AS (
  SELECT
    profile_id_i64,
    leaderboard,
    started_timestamp,
    game_id,
    old_rating,
    new_rating,
    LAG(new_rating) OVER (
      PARTITION BY profile_id_i64, leaderboard
      ORDER BY started_timestamp, game_id
    ) AS prev_new_rating,
    LAG(started_timestamp) OVER (
      PARTITION BY profile_id_i64, leaderboard
      ORDER BY started_timestamp, game_id
    ) AS prev_started_timestamp
  FROM deduped
),
pairs AS (
  SELECT *,
    CASE WHEN old_rating = prev_new_rating THEN 1 ELSE 0 END AS agreed,
    EPOCH(started_timestamp - prev_started_timestamp) / 86400.0 AS gap_days
  FROM with_lag
  WHERE prev_new_rating IS NOT NULL
),
bucketed AS (
  SELECT
    CASE
      WHEN gap_days < 1   THEN '<1d'
      WHEN gap_days < 7   THEN '1-7d'
      WHEN gap_days < 30  THEN '7-30d'
      ELSE                     '>30d'
    END AS time_gap_bucket,
    agreed
  FROM pairs
)
SELECT
  time_gap_bucket,
  COUNT(*)           AS n_pairs,
  AVG(agreed::DOUBLE) AS agreement_rate
FROM bucketed
GROUP BY time_gap_bucket
ORDER BY MIN(
  CASE time_gap_bucket
    WHEN '<1d'   THEN 0
    WHEN '1-7d'  THEN 1
    WHEN '7-30d' THEN 2
    ELSE              3
  END
)
"""

gap_df = con.execute(SQL_TIME_GAP_STRATIFICATION).df()
print("Per-time-gap-bucket stratification (random_map):")
print(gap_df.to_string(index=False))

per_time_gap_agreement = {
    row["time_gap_bucket"]: float(row["agreement_rate"])
    for _, row in gap_df.iterrows()
}

per_leaderboard_agreement = {
    row["leaderboard"]: float(row["agreement_rate"])
    for _, row in lb_result.iterrows()
}

print()
print("Per-leaderboard summary:")
for lb, rate in per_leaderboard_agreement.items():
    gate = "PASS" if rate >= 0.90 else "FAIL"
    print(f"  {lb:20s}: {rate:.4f}  [{gate}]")

print()
print("Per-time-gap-bucket summary:")
for bucket, rate in per_time_gap_agreement.items():
    gate = "PASS" if rate >= 0.90 else "FAIL"
    print(f"  {bucket:8s}: {rate:.4f}  [{gate}]")

Per-time-gap-bucket stratification (random_map):
time_gap_bucket  n_pairs  agreement_rate
            <1d 29021397        0.943973
           1-7d  4661976        0.859000
          7-30d  1057793        0.707557
           >30d   534031        0.634484

Per-leaderboard summary:
  team_random_map     : 0.8601  [FAIL]
  random_map          : 0.9210  [PASS]
  co_team_random_map  : 0.7867  [FAIL]
  co_random_map       : 0.8552  [FAIL]

Per-time-gap-bucket summary:
  <1d     : 0.9440  [PASS]
  1-7d    : 0.8590  [FAIL]
  7-30d   : 0.7076  [FAIL]
  >30d    : 0.6345  [FAIL]


## Verdict Cell

3-gate PASS (post-Mode-A stratification-conflict rule):
(a) primary scope `agreement_rate >= 0.95` AND `max_disagreement < 50`
(b) every leaderboard stratum `agreement_rate >= 0.90`
(c) every time-gap-bucket `agreement_rate >= 0.90`
ALL THREE required for PASS.
MARGINAL if ONLY gate (a) fails because primary ∈ [0.90, 0.95) and (b)+(c) pass.
FAIL otherwise.
CATASTROPHIC_HALT if primary `agreement_rate < 0.80` → halt, no artifact export.

In [10]:
THRESHOLD_PRIMARY = 0.95
THRESHOLD_STRATUM = 0.90
THRESHOLD_MAX_DISAGREEMENT = 50
THRESHOLD_CATASTROPHIC = 0.80

# Gate (a)
gate_a_rate = agreement_rate_primary >= THRESHOLD_PRIMARY
gate_a_magnitude = max_disagreement_primary < THRESHOLD_MAX_DISAGREEMENT
gate_a = gate_a_rate and gate_a_magnitude

# Gate (b)
lb_failures = {lb: r for lb, r in per_leaderboard_agreement.items() if r < THRESHOLD_STRATUM}
gate_b = len(lb_failures) == 0

# Gate (c)
gap_failures = {b: r for b, r in per_time_gap_agreement.items() if r < THRESHOLD_STRATUM}
gate_c = len(gap_failures) == 0

print("=== VERDICT EVALUATION ===")
print(f"Gate (a) primary rate:       {agreement_rate_primary:.4f} >= {THRESHOLD_PRIMARY}  -> {'PASS' if gate_a_rate else 'FAIL'}")
print(f"Gate (a) max disagreement:   {max_disagreement_primary} < {THRESHOLD_MAX_DISAGREEMENT}  -> {'PASS' if gate_a_magnitude else 'FAIL'}")
print(f"Gate (a) combined:           {'PASS' if gate_a else 'FAIL'}")
print(f"Gate (b) all-leaderboard:    {'PASS' if gate_b else 'FAIL'} (failures: {lb_failures})")
print(f"Gate (c) all-time-gap:       {'PASS' if gate_c else 'FAIL'} (failures: {gap_failures})")
print()

if agreement_rate_primary < THRESHOLD_CATASTROPHIC:
    verdict = "CATASTROPHIC_HALT"
    print("CATASTROPHIC_HALT: agreement_rate_primary < 0.80")
    print("Artifact export SKIPPED. Escalate to user before proceeding.")
elif gate_a and gate_b and gate_c:
    verdict = "PASS"
elif (not gate_a_rate) and gate_a_magnitude and gate_b and gate_c:
    # Only gate (a) rate fails, magnitude ok, strata ok
    verdict = "MARGINAL"
else:
    verdict = "FAIL"

print(f"VERDICT: {verdict}")

# Interpretation
if verdict == "PASS":
    print()
    print("Interpretation: API honours the old/new-rating convention on random_map")
    print("(primary) with consistent behaviour across all leaderboards and time-gap")
    print("buckets. Structural evidence (matches_1v1_clean.yaml:excluded_columns) +")
    print("empirical evidence (this step) close the §3 deferral flag.")
elif verdict == "MARGINAL":
    print()
    print("Interpretation: primary scope ∈ [0.90, 0.95). PRE-GAME classification")
    print("is retained but with explicit caveat; thesis §4.2.3 should hedge.")
elif verdict == "FAIL":
    print()
    print("Interpretation: one or more gates failed beyond MARGINAL tolerance.")
    print("The PRE-GAME classification is structurally supported but empirically")
    print("uncertain. Three follow-up candidates:")
    print("  1. Retain with caveat (agreement ∈ [0.90, 0.95) for primary only)")
    print("  2. Demote to CONDITIONAL_PRE_GAME with filter investigation")
    print("  3. Filter out disagreeing player-match pairs in Phase 02 feature engineering")
    print()
    print("Key observation: disagreement concentrates in longer-gap pairs (1-7d, 7-30d, >30d),")
    print("consistent with rating resets, seasonal updates, or players returning after long")
    print("breaks. Short-gap (<1d) agreement rate is high (>0.94), confirming the API")
    print("convention holds for dense play patterns.")

=== VERDICT EVALUATION ===
Gate (a) primary rate:       0.9210 >= 0.95  -> FAIL
Gate (a) max disagreement:   1118 < 50  -> FAIL
Gate (a) combined:           FAIL
Gate (b) all-leaderboard:    FAIL (failures: {'team_random_map': 0.8601277087250514, 'co_team_random_map': 0.7867429143732843, 'co_random_map': 0.8552393158798064})
Gate (c) all-time-gap:       FAIL (failures: {'1-7d': 0.8590001321328123, '7-30d': 0.7075571496502624, '>30d': 0.634483765923701})

VERDICT: FAIL

Interpretation: one or more gates failed beyond MARGINAL tolerance.
The PRE-GAME classification is structurally supported but empirically
uncertain. Three follow-up candidates:
  1. Retain with caveat (agreement ∈ [0.90, 0.95) for primary only)
  2. Demote to CONDITIONAL_PRE_GAME with filter investigation
  3. Filter out disagreeing player-match pairs in Phase 02 feature engineering

Key observation: disagreement concentrates in longer-gap pairs (1-7d, 7-30d, >30d),
consistent with rating resets, seasonal updates, or pla

## Artifact Export Cell

Skipped if CATASTROPHIC_HALT.

In [11]:
if verdict == "CATASTROPHIC_HALT":
    print("Artifact export SKIPPED due to CATASTROPHIC_HALT.")
else:
    # Build artifact payload
    artifact = {
        "n_pairs_primary_scope": n_pairs_primary,
        "n_pairs_total_all_leaderboards": n_pairs_total,
        "agreement_rate_primary": round(agreement_rate_primary, 6),
        "max_disagreement_primary": max_disagreement_primary,
        "median_disagreement_disagreeing_pairs": median_dis_disagreeing,
        "wilson_ci_95_low": round(wilson_ci_low, 6),
        "wilson_ci_95_high": round(wilson_ci_high, 6),
        "per_leaderboard_agreement": {k: round(v, 6) for k, v in per_leaderboard_agreement.items()},
        "per_time_gap_agreement": {k: round(v, 6) for k, v in per_time_gap_agreement.items()},
        "tie_rate": float(tie_rate),
        "duplicate_rows_collapsed": int(extra_rows) if extra_rows else 0,
        "verdict": verdict,
        "hypothesis_thresholds": {
            "agreement_primary_min": THRESHOLD_PRIMARY,
            "max_disagreement_max": THRESHOLD_MAX_DISAGREEMENT,
            "stratum_min": THRESHOLD_STRATUM,
            "catastrophic_halt_min": THRESHOLD_CATASTROPHIC,
        },
        "threshold_rationale": (
            "DS-AOESTATS-02 2-order-of-magnitude factor over 0.03% NULLIF tolerance; "
            "50 rating unit approximate Elo K=40 scale"
        ),
        "ci_method": "wilson",
        "audit_date": str(date.today()),
        "gate_a_rate_pass": gate_a_rate,
        "gate_a_magnitude_pass": gate_a_magnitude,
        "gate_b_pass": gate_b,
        "gate_c_pass": gate_c,
        "lb_stratum_failures": lb_failures,
        "gap_bucket_failures": gap_failures,
    }

    json_path.write_text(json.dumps(artifact, indent=2))
    print(f"JSON written: {json_path}")

    # Build MD artifact
    lb_table_rows = "\n".join(
        f"| {r['leaderboard']} | {r['n_pairs']:,} | {r['agreement_rate']:.4f} | "
        f"{'PASS' if r['agreement_rate'] >= THRESHOLD_STRATUM else 'FAIL'} |"
        for _, r in lb_result.iterrows()
    )
    gap_table_rows = "\n".join(
        f"| {row['time_gap_bucket']} | {row['n_pairs']:,} | {row['agreement_rate']:.4f} | "
        f"{'PASS' if row['agreement_rate'] >= THRESHOLD_STRATUM else 'FAIL'} |"
        for _, row in gap_df.iterrows()
    )

    if verdict == "PASS":
        interp_text = (
            "API honours the old/new-rating convention on `random_map` (primary) with consistent "
            "behaviour across all leaderboards. Structural evidence "
            "(`matches_1v1_clean.yaml:excluded_columns`) + empirical evidence (this step) close "
            "the §3 deferral flag."
        )
    elif verdict == "MARGINAL":
        interp_text = (
            "Primary scope agreement rate {:.4f} ∈ [0.90, 0.95). PRE-GAME classification "
            "is retained but with explicit caveat; thesis §4.2.3 should hedge. "
            "Structural evidence (`matches_1v1_clean.yaml:excluded_columns`) partially "
            "supports the classification; empirical evidence is inconclusive at the 0.95 standard."
        ).format(agreement_rate_primary)
    else:
        interp_text = (
            f"Gates failed: gate(a)-rate={gate_a_rate}, gate(a)-magnitude={gate_a_magnitude}, "
            f"gate(b)={gate_b}, gate(c)={gate_c}. "
            "The PRE-GAME classification is structurally supported (VIEW exclusion of `new_rating`) "
            "but empirically uncertain. Disagreement concentrates in longer-gap pairs "
            "(1-7d: {:.3f}, 7-30d: {:.3f}, >30d: {:.3f}), consistent with rating resets or "
            "seasonal updates. Short-gap (<1d) agreement {:.4f} confirms the convention holds "
            "for dense play. Three follow-up candidates: "
            "(1) retain with caveat if primary is deemed acceptable for thesis purposes; "
            "(2) demote to CONDITIONAL_PRE_GAME pending investigation of reset mechanisms; "
            "(3) filter disagreeing pairs in Phase 02 feature engineering."
        ).format(
            per_time_gap_agreement.get("1-7d", 0),
            per_time_gap_agreement.get("7-30d", 0),
            per_time_gap_agreement.get(">30d", 0),
            per_time_gap_agreement.get("<1d", 0),
        )

    md_content = f"""# Audit: old_rating PRE-GAME Temporal Consistency (Step 01_04_06)

**Dataset:** aoestats
**Date:** {date.today().isoformat()}
**Verdict:** {verdict}

---

## §1 Scope and Method

**Hypothesis:** On `leaderboard = 'random_map'` (primary), for consecutive matches t, t+1
of the same `profile_id` ordered by `(started_timestamp, game_id)` within the same leaderboard,
`players_raw.old_rating[t+1] == players_raw.new_rating[t]` at `agreement_rate >= 0.95`
AND `max(|old_rating[t+1] - new_rating[t]|) < 50` rating units.
Per-leaderboard strata pass at >= 0.90. Per-time-gap-bucket strata pass at >= 0.90.

**3-gate falsifier:**
- Gate (a): primary scope `agreement_rate >= 0.95` AND `max_disagreement < 50`
- Gate (b): every leaderboard stratum `agreement_rate >= 0.90`
- Gate (c): every time-gap-bucket `agreement_rate >= 0.90`
- CATASTROPHIC_HALT: `agreement_rate < 0.80`

**Threshold rationale (I7):**
- 0.95 primary threshold: DS-AOESTATS-02 documents NULLIF cleaning-loss of ~0.03% for
  `old_rating=0` sentinel rows. Agreement rate below 0.95 (>5% disagreement) would exceed
  this tolerance by 2 orders of magnitude, making the PRE-GAME classification materially
  unsound.
- 50-unit magnitude threshold: Elo K-factor for aoestats is approximately 20-40 rating points
  per match. Disagreements > 50 units indicate systematic drift beyond normal Elo updates.
- 0.90 stratum threshold: relaxed for sub-strata to accommodate smaller n and
  leaderboard-specific factors (co-op modes, team ladders have different population sizes).

**CAST discipline (DS-AOESTATS-IDENTITY-04):** `profile_id` stored as DOUBLE; all values
below 2^53, so `CAST(profile_id AS BIGINT)` is lossless. Defined as `profile_id_i64` in all CTEs.

**LAG partition (BLOCKER 1 fix):** AoE2 ratings are per-leaderboard independent systems.
Window: `PARTITION BY (profile_id_i64, leaderboard) ORDER BY (started_timestamp, game_id)`.

---

## §2 SQL Verbatim (I6)

### Tie-rate pre-flight

```sql
{SQL_TIE_RATE}
```

### Duplicate-row pre-flight

```sql
{SQL_DUPLICATES}
```

### Pair construction and per-leaderboard agreement

```sql
{SQL_PAIRS_PER_LEADERBOARD}
```

### Primary scope agreement and disagreement magnitude

```sql
{SQL_PRIMARY_AGREEMENT}
```

### Per-time-gap-bucket stratification

```sql
{SQL_TIME_GAP_STRATIFICATION}
```

---

## §3 Results

**Pre-flight:**
- Tie rate: {tie_rate:.6%} ({tied_groups} tied groups / {total_groups:,} total) — well below 1% threshold
- Duplicate (game_id, profile_id_i64) extra rows: {extra_rows if extra_rows else 0} — DISTINCT applied

**Primary scope (random_map):**
| Metric | Value |
|--------|-------|
| n_pairs | {n_pairs_primary:,} |
| agreement_rate | {agreement_rate_primary:.6f} |
| max_disagreement | {max_disagreement_primary} rating units |
| median_disagreement (disagreeing only) | {median_dis_disagreeing} |
| Wilson CI 95% low | {wilson_ci_low:.6f} |
| Wilson CI 95% high | {wilson_ci_high:.6f} |
| Total pairs (all leaderboards) | {n_pairs_total:,} |

**Per-leaderboard agreement:**
| Leaderboard | n_pairs | agreement_rate | Gate (b) |
|-------------|---------|----------------|----------|
{lb_table_rows}

**Per-time-gap-bucket (random_map):**
| Bucket | n_pairs | agreement_rate | Gate (c) |
|--------|---------|----------------|----------|
{gap_table_rows}

---

## §4 Verdict

| Gate | Criterion | Value | Status |
|------|-----------|-------|--------|
| (a) rate | primary agreement_rate >= 0.95 | {agreement_rate_primary:.4f} | {'PASS' if gate_a_rate else 'FAIL'} |
| (a) magnitude | max_disagreement < 50 units | {max_disagreement_primary} | {'PASS' if gate_a_magnitude else 'FAIL'} |
| (b) strata | all leaderboards >= 0.90 | failures: {list(lb_failures.keys())} | {'PASS' if gate_b else 'FAIL'} |
| (c) time-gap | all time-gap buckets >= 0.90 | failures: {list(gap_failures.keys())} | {'PASS' if gate_c else 'FAIL'} |

**Overall verdict: {verdict}**

---

## §5 Interpretation

{interp_text}
"""

    md_path.write_text(md_content)
    print(f"MD written: {md_path}")

    print()
    print("=== FINAL SUMMARY ===")
    print(f"Verdict: {verdict}")
    print(f"Primary agreement: {agreement_rate_primary:.4f}")
    print(f"Max disagreement: {max_disagreement_primary} rating units")
    print(f"Wilson CI 95%: [{wilson_ci_low:.4f}, {wilson_ci_high:.4f}]")
    print(f"Gate (a) rate: {'PASS' if gate_a_rate else 'FAIL'}")
    print(f"Gate (a) magnitude: {'PASS' if gate_a_magnitude else 'FAIL'}")
    print(f"Gate (b) leaderboard strata: {'PASS' if gate_b else 'FAIL'}")
    print(f"Gate (c) time-gap buckets: {'PASS' if gate_c else 'FAIL'}")

JSON written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/04_cleaning/01_04_06_old_rating_temporal_audit.json
MD written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/04_cleaning/01_04_06_old_rating_temporal_audit.md

=== FINAL SUMMARY ===
Verdict: FAIL
Primary agreement: 0.9210
Max disagreement: 1118 rating units
Wilson CI 95%: [0.9209, 0.9211]
Gate (a) rate: FAIL
Gate (a) magnitude: FAIL
Gate (b) leaderboard strata: FAIL
Gate (c) time-gap buckets: FAIL
